# BrainDecoding — Pallier2025 统一 `causal_fast_audit` 词级主表与核心消融云端补跑流水线

本 Notebook 用于在 Google Colab GPU 环境下，以**严格的单一锁定口径**对 `BrainDecoding/runs/pallier2025` 的 **6 组**词级主表基线与核心消融模型补跑统一的 **`causal_fast_audit`** 多种子测试集评估 (`Seeds [0, 1, 2]`, `split = test`)，对齐已锁定的 **主方法锚点** (`2_lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window / prefix_revision`)。

## 四项强制锁定参数（全部 6 组 + 锚点 1:1 统一）
1. **锁定同交集查询池 (`common_subset`)**：统一只评测对齐 Stage 2.3 的 **$14,590$ 全词表事件 / $7,440$ 条 Top-50 词表查询**（每被试 $1,459$ / $744$），Clean 与负对照严格使用同一查询池（`word` / `context` 通过 `--common-horizon-delay 2` 对齐，且绝不泄露 $8,930$ 未过滤查询）；
2. **锁定 Pre-Session 因果移位池 (`preroll + M=4`)**：`shift_mode = preroll`、`preroll_seconds = 45.0`、`shift_seconds = [-10, -20, -30, -40]`（不混合历史圆周移位）；
3. **锁定极简负对照与在线因果共模扣除**：`cms_mode = causal_ema`、`ema_decay = 0.99`、`mu0_source = train_readonly_fallback`、对照方法 `raw,causal_cms:eta=0.5`，**不运行** 24 行非因果子空间探索 sweep；
4. **独立隔离输出目录 (`causal_fast_audit/`)**：全部落盘至 `<exp_tag>/causal_fast_audit/<stage>/all-n10/`，**不覆盖**任何历史全量报告，也**绝不重跑或覆盖**已锁定的主方法锚点 (`2_lr4e-4_..._history_window/causal_fast_audit/prefix_revision/all-n10/`)。

| 单元格 | 任务角色 | `exp_tag / stage` | 配置文件 | 共同查询裁剪 |
| --- | --- | --- | --- | --- |
| **单元格 2** | 环境初始化 & 锁参执行器 | — | — | `features_meg.tar` + 锚点/检查点只读预检 |
| **单元格 4** | 主表基线 1 (Stage 1 Word) | `lr4e-4_bs256 / word` | `pallier2025_word_encoder.yaml` | `--common-horizon-delay 2` |
| **单元格 6** | 主表基线 2a (Joint Context) | `stage1_lr4e-4_bs256_stage2_lr1e-4_bs128 / context` | `pallier2025_context_transformer.yaml` | `--common-horizon-delay 2` |
| **单元格 8** | 主表基线 2b (+ Freeze CNN) | `stage1_freeze_bs256_stage2_lr1e-4_bs128 / context` | `pallier2025_context_transformer.yaml` | `--common-horizon-delay 2` |
| **单元格 10** | 主表基线 3 (+ Causal CMS r=0) | `lr4e-4_..._history_window / context` | `pallier2025_siamese_debiased_context.yaml` | `--common-horizon-delay 2` |
| **单元格 12** | 核心消融 1 (Stage 2.2 Local r=2) | `lr4e-4_..._history_window / revision` | `pallier2025_siamese_debiased_revision.yaml` | 原生 $R=2$ 掩码 (`N=14,590`) |
| **单元格 14** | 核心消融 2 (Stage 2.3 w/o L_delta-null) | `lr4e-4_..._history_window / prefix_revision` | `pallier2025_siamese_debiased_prefix_revision.yaml` | 原生 $R=2$ 掩码 (`N=14,590`) |
| **单元格 16** | 一键批处理（可选连续执行任务 1–6） | 全部 6 组 | — | 自动跳过已完成并校验锁参一致的组 |
| **单元格 18** | 锚点只读联检 + 7 组总表导出 | `causal_fast_audit_summary/` | — | 导出 `.csv` / `.md` / `.json` 并打印对比总表 |


## 单元格 1：云端环境初始化、Google Drive 挂载、`features_meg.tar` 解包与 `causal_fast_audit` 锁参执行器


In [ ]:
# 单元格 1：挂载 Google Drive、同步最新代码、初始化环境、解包 features_meg.tar 并定义锁参评估函数
!nvidia-smi

import csv
import json
import os
import subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('非 Colab 环境，跳过 drive.mount')

# ==================== 1. 路径与仓库配置 ====================
GIT_REPO_URL = 'https://github.com/wdongdong0708-jpg/BrainDecoding2.git'
GIT_BRANCH = 'master'
DRIVE_BASE_DIR = '/content/drive/MyDrive/BrainDecoding'
DATASET = 'pallier2025'
MODALITY = 'meg'

# 七大目标实验目录标签（6 组待补跑 + 1 组已锁定主方法锚点）
TAG_WORD_ONLY = 'lr4e-4_bs256'
TAG_JOINT_UNFROZEN_CTX = 'stage1_lr4e-4_bs256_stage2_lr1e-4_bs128'
TAG_FREEZE_CNN_CTX = 'stage1_freeze_bs256_stage2_lr1e-4_bs128'
TAG_SIAMESE_HISTORY = 'lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window'
TAG_PREFIX_FULL = '2_lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window'

CODE_DIR = '/content/code'
VENV_DIR = '/content/proj-venv'
PYTHON = f'{VENV_DIR}/bin/python'
LINKS_DIR = '/content/studies_links'
LOCAL_CACHE_DIR = '/content/local_cache'

DRIVE_DATA_DIR = f'{DRIVE_BASE_DIR}/data'
DRIVE_ARCHIVE_DIR = f'{DRIVE_BASE_DIR}/archives'
DRIVE_RESULTS_DIR = f'{DRIVE_BASE_DIR}/runs'
DRIVE_MODELS_DIR = f'{DRIVE_BASE_DIR}/models'

for folder in (
    LINKS_DIR,
    LOCAL_CACHE_DIR,
    DRIVE_DATA_DIR,
    DRIVE_ARCHIVE_DIR,
    DRIVE_RESULTS_DIR,
    DRIVE_MODELS_DIR,
):
    Path(folder).mkdir(parents=True, exist_ok=True)

os.environ['BRAINDECODING_DATA_DIR'] = LINKS_DIR
os.environ['BRAINDATA_ROOT'] = LINKS_DIR
os.environ['BRAINDECODING_CACHE_DIR'] = LOCAL_CACHE_DIR
os.environ['BRAINDECODING_RESULTS_DIR'] = DRIVE_RESULTS_DIR
os.environ['BRAINDECODING_MODEL_DIR'] = DRIVE_MODELS_DIR

# ==================== 2. 锁定 causal_fast_audit 评估参数 ====================
EVAL_SUBDIR = 'causal_fast_audit'
CTRL_CAUSAL_CMS = 'configs/controls/common_mode_subtraction/causal_common_mode_subtraction_control.yaml'
LOCKED_WINDOW_SECONDS = 1.0
LOCKED_SEEDS = '0,1,2'
LOCKED_SPLIT = 'test'
LOCKED_SUBJECT = 'all'
LOCKED_SHIFT_MODE = 'preroll'
LOCKED_PREROLL_SECONDS = 45.0
LOCKED_SHIFT_SECONDS = [-10.0, -20.0, -30.0, -40.0]
LOCKED_SHIFT_ARG = '-10,-20,-30,-40'
LOCKED_CMS_MODE = 'causal_ema'
LOCKED_CMS_EMA_DECAY = 0.99
LOCKED_SUBTRACTION_ETAS = '0.5'
LOCKED_CONTROL_METHODS = ['raw', 'causal_cms:eta=0.5']
LOCKED_CONTROL_METHOD_ARG = 'raw,causal_cms:eta=0.5'
LOCKED_CONTROL_METRIC_ARG = 'vocab50_macro_top10,vocab50_macro_top1,full_vocab_macro_top10,full_vocab_macro_top1'
LOCKED_COMMON_HORIZON_DELAY = 2
LOCKED_FULL_QUERY_COUNT = 14590
LOCKED_N50_QUERY_COUNT = 7440

# 默认若云盘已有通过锁参校验的完整 3 种子结果则跳过；设为 True 可强制覆盖重跑这 6 组（锚点永远只读）
FORCE_RERUN = False

# ==================== 3. 任务注册表 ====================
CAUSAL_FAST_AUDIT_TASKS = [
    {
        'task_id': 'baseline_1_word',
        'role': '主表基线 1 (Stage 1 冻结词编码器)',
        'exp_tag': TAG_WORD_ONLY,
        'word_exp_tag': TAG_WORD_ONLY,
        'stage': 'word',
        'config': 'configs/training/pallier2025_word_encoder.yaml',
        'common_horizon_delay': LOCKED_COMMON_HORIZON_DELAY,
        'legacy_ref': '40.64 ± 0.15% / 旧M6 19.88%~20.28%',
    },
    {
        'task_id': 'baseline_2a_joint_ctx',
        'role': '主表基线 2a (未去偏全句上下文·微调 CNN)',
        'exp_tag': TAG_JOINT_UNFROZEN_CTX,
        'word_exp_tag': TAG_WORD_ONLY,
        'stage': 'context',
        'config': 'configs/training/pallier2025_context_transformer.yaml',
        'common_horizon_delay': LOCKED_COMMON_HORIZON_DELAY,
        'legacy_ref': '68.43 ± 0.43% / 旧M6 60.34%~61.73%',
    },
    {
        'task_id': 'baseline_2b_freeze_ctx',
        'role': '主表基线 2b (未去偏全句上下文·冻结 Stage 1)',
        'exp_tag': TAG_FREEZE_CNN_CTX,
        'word_exp_tag': TAG_WORD_ONLY,
        'stage': 'context',
        'config': 'configs/training/pallier2025_context_transformer.yaml',
        'common_horizon_delay': LOCKED_COMMON_HORIZON_DELAY,
        'legacy_ref': '64.34 ± 0.91% / 旧M6 50.12 ± 3.19%',
    },
    {
        'task_id': 'baseline_3_debiased_ctx',
        'role': '主表基线 3 (孪生去偏全句非因果参考)',
        'exp_tag': TAG_SIAMESE_HISTORY,
        'word_exp_tag': TAG_WORD_ONLY,
        'stage': 'context',
        'config': 'configs/training/siamese_debiased/pallier2025_siamese_debiased_context.yaml',
        'common_horizon_delay': LOCKED_COMMON_HORIZON_DELAY,
        'legacy_ref': '57.95 ± 2.51% / 旧M6 22.04%~22.43%',
    },
    {
        'task_id': 'ablation_1_local_revision',
        'role': '核心消融 1 (Stage 2.2 局部固定窗 R=2 修订)',
        'exp_tag': TAG_SIAMESE_HISTORY,
        'word_exp_tag': TAG_WORD_ONLY,
        'stage': 'revision',
        'config': 'configs/training/siamese_debiased/pallier2025_siamese_debiased_revision.yaml',
        'common_horizon_delay': None,
        'legacy_ref': '61.46 ± 0.88% / 旧M6 30.25 ± 1.32%',
    },
    {
        'task_id': 'ablation_2_prefix_wo_delta',
        'role': '核心消融 2 (Stage 2.3 无 L_delta-null 前缀修订)',
        'exp_tag': TAG_SIAMESE_HISTORY,
        'word_exp_tag': TAG_WORD_ONLY,
        'stage': 'prefix_revision',
        'config': 'configs/training/siamese_debiased/pallier2025_siamese_debiased_prefix_revision.yaml',
        'common_horizon_delay': None,
        'legacy_ref': '57.90 ± 1.18% / 旧M6 24.76 ± 0.74%',
    },
]

LOCKED_ANCHOR_SPEC = {
    'task_id': 'main_method_anchor',
    'role': '主方法锚点 (Stage 2.3 Full Prefix-Revision · 只读)',
    'exp_tag': TAG_PREFIX_FULL,
    'word_exp_tag': TAG_WORD_ONLY,
    'stage': 'prefix_revision',
    'legacy_ref': '已锁定 Clean 58.70 ± 1.02% / Ctrl 23.65 ± 0.17%',
}


def run_cmd(cmd: str, env_overrides: dict | None = None) -> None:
    """在 /content/code 下执行命令并实时流式打印输出，遇到非零退出码立即抛错。"""
    print(f'\n>>> {cmd}', flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    if env_overrides:
        for k, v in env_overrides.items():
            if v is None:
                env.pop(k, None)
            else:
                env[k] = str(v)
    proc = subprocess.Popen(
        cmd,
        shell=True,
        cwd=CODE_DIR if Path(CODE_DIR).is_dir() else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f'命令执行失败 (exit={ret}): {cmd}')


# ==================== 4. 同步代码、初始化虚拟环境并解包缓存 ====================
if not Path(f'{CODE_DIR}/.git').is_dir():
    run_cmd(f'git clone -b {GIT_BRANCH} {GIT_REPO_URL} {CODE_DIR}')
else:
    run_cmd(f'git -C {CODE_DIR} fetch origin && git -C {CODE_DIR} checkout {GIT_BRANCH} && git -C {CODE_DIR} pull --ff-only origin {GIT_BRANCH}')

run_cmd(
    f'python3 -u {CODE_DIR}/scripts/colab_setup.py '
    f'--venv-dir {VENV_DIR} '
    f'--project-root {CODE_DIR} '
    f'--drive-data-dir {DRIVE_DATA_DIR} '
    f'--links-dir {LINKS_DIR} '
    f'--local-cache-dir {LOCAL_CACHE_DIR}'
)

run_cmd(
    f'{PYTHON} -u scripts/cache_archive.py unpack '
    f'--archive-dir {DRIVE_ARCHIVE_DIR} '
    f'--cache-dir {LOCAL_CACHE_DIR} '
    f'--modality {MODALITY} '
    f'--skip-existing'
)


def causal_audit_stage_dir(exp_tag: str, stage: str) -> Path:
    return Path(DRIVE_RESULTS_DIR) / DATASET / exp_tag / EVAL_SUBDIR / stage / 'all-n10'


def verify_causal_fast_audit_summary(summary_json_path: Path) -> tuple[bool, list[str], dict | None]:
    """校验 multi_seed_evaluation_test.json 是否严格满足 4 项锁定口径。"""
    errors: list[str] = []
    if not summary_json_path.is_file():
        return False, [f'缺失文件: {summary_json_path}'], None
    try:
        data = json.loads(summary_json_path.read_text(encoding='utf-8'))
    except Exception as exc:
        return False, [f'JSON 解析失败: {exc}'], None

    if int(data.get('seed_count', 0) or 0) != 3 or list(data.get('seeds') or []) != [0, 1, 2]:
        errors.append(f"seeds 异常: seed_count={data.get('seed_count')}, seeds={data.get('seeds')}")

    vsa = data.get('vocabulary_sweep_aggregate') or {}
    n50_q = int(((vsa.get('N50') or {}).get('query_count')) or 0)
    full_q = int(((vsa.get('full_vocab') or {}).get('query_count')) or 0)
    if n50_q != LOCKED_N50_QUERY_COUNT:
        errors.append(f'N50 query_count={n50_q} != {LOCKED_N50_QUERY_COUNT}')
    if full_q != LOCKED_FULL_QUERY_COUNT:
        errors.append(f'full_vocab query_count={full_q} != {LOCKED_FULL_QUERY_COUNT}')

    sel_methods = list(data.get('selected_control_methods') or [])
    if sel_methods != LOCKED_CONTROL_METHODS:
        errors.append(f'selected_control_methods={sel_methods} != {LOCKED_CONTROL_METHODS}')

    cms_agg = data.get('common_mode_subtraction_aggregate') or {}
    if not cms_agg.get('enabled') or str(cms_agg.get('cms_mode', '')).lower() != LOCKED_CMS_MODE:
        errors.append(f"cms_mode 异常: {cms_agg.get('cms_mode')}")

    sub_agg = data.get('subspace_debiasing_aggregate') or {}
    if bool(sub_agg.get('enabled', False)):
        errors.append('subspace_debiasing_aggregate.enabled 必须为 False（禁止混入 24 行非因果子空间表）')

    ncm = data.get('negative_control_methods_aggregate') or {}
    raw_agg = ncm.get('raw') or {}
    shifts = [round(float(x), 4) for x in (raw_agg.get('shift_seconds') or [])]
    if shifts != LOCKED_SHIFT_SECONDS:
        errors.append(f'shift_seconds={shifts} != {LOCKED_SHIFT_SECONDS}')

    seed_results = list(data.get('seed_results') or [])
    if len(seed_results) != 3:
        errors.append(f'seed_results 数量={len(seed_results)} != 3')
    for idx, sr in enumerate(seed_results):
        nc = sr.get('negative_controls') or {}
        spec = nc.get('specification') or {}
        if str(spec.get('shift_mode', '')).lower() != LOCKED_SHIFT_MODE:
            errors.append(f"seed[{idx}] shift_mode={spec.get('shift_mode')} != {LOCKED_SHIFT_MODE}")
        if abs(float(spec.get('preroll_seconds') or 0.0) - LOCKED_PREROLL_SECONDS) > 1e-3:
            errors.append(f"seed[{idx}] preroll_seconds={spec.get('preroll_seconds')} != {LOCKED_PREROLL_SECONDS}")
        sr_n50_q = int((((sr.get('metrics') or {}).get('vocabulary_sweep') or {}).get('N50') or {}).get('query_count') or 0)
        if sr_n50_q != LOCKED_N50_QUERY_COUNT:
            errors.append(f'seed[{idx}] N50 query_count={sr_n50_q} != {LOCKED_N50_QUERY_COUNT}')
        cms_path = sr.get('exploratory_common_mode_subtraction_json')
        if cms_path and Path(str(cms_path)).is_file():
            try:
                cms_d = json.loads(Path(str(cms_path)).read_text(encoding='utf-8'))
                mu0_src = (cms_d.get('formula') or {}).get('mu0_source') or cms_d.get('mu0_source')
                if mu0_src is not None and str(mu0_src) in ('zeros_fallback', 'cold_zero_init'):
                    errors.append(f'seed[{idx}] mu0_source={mu0_src}（疑似遗漏 train_readonly_fallback 校准）')
            except Exception:
                pass

    return len(errors) == 0, errors, data


def run_causal_fast_audit_task(task: dict, *, force: bool | None = None) -> dict:
    """按严格锁定参数执行单组 causal_fast_audit 多种子评估，并在完成前后执行校验。"""
    do_force = FORCE_RERUN if force is None else bool(force)
    exp_tag = task['exp_tag']
    word_exp_tag = task['word_exp_tag']
    stage = task['stage']
    cfg_rel = task['config']
    delay = task['common_horizon_delay']

    # 1. 只读校验 3 颗检查点与配置文件存在性
    cfg_abs = Path(CODE_DIR) / cfg_rel
    if not cfg_abs.is_file():
        raise FileNotFoundError(f'缺少配置文件: {cfg_abs}')
    for sd in (0, 1, 2):
        ckpt = Path(DRIVE_RESULTS_DIR) / DATASET / exp_tag / stage / 'all-n10' / f'seed-{sd:03d}' / 'best.ckpt'
        if not ckpt.is_file():
            raise FileNotFoundError(f'缺少检查点: {ckpt}')

    out_stage_dir = causal_audit_stage_dir(exp_tag, stage)
    summary_json = out_stage_dir / f'multi_seed_evaluation_{LOCKED_SPLIT}.json'

    if not do_force and summary_json.is_file():
        ok, errs, existing = verify_causal_fast_audit_summary(summary_json)
        if ok and existing is not None:
            print(f"[SKIP] {task['role']} 已存在且严格通过锁参校验 -> {summary_json}")
            return existing
        print(f"[RE-RUN] 检测到既有结果未完全对齐锁定参数，将重跑并覆盖 {EVAL_SUBDIR}: {errs}")

    print('=' * 92)
    print(f" [TASK] {task['role']}")
    print(f" exp_tag={exp_tag} | stage={stage} | config={cfg_rel} | common_horizon_delay={delay}")
    print('=' * 92, flush=True)

    cmd_parts = [
        f'{PYTHON} -u -m braindecoding eval-run',
        f'--config {cfg_rel}',
        f'--control-config {CTRL_CAUSAL_CMS}',
        f'--eval-subdir {EVAL_SUBDIR}',
        f'--exp-tag {exp_tag}',
        f'--word-exp-tag {word_exp_tag}',
        f'--stage {stage}',
        f'--window-seconds {LOCKED_WINDOW_SECONDS}',
        f'--seed {LOCKED_SEEDS}',
        f'--split {LOCKED_SPLIT}',
        '--require-cache',
        f'--subject {LOCKED_SUBJECT}',
        f'--shift-mode {LOCKED_SHIFT_MODE}',
        f'--preroll-seconds {LOCKED_PREROLL_SECONDS}',
        f'--shift-seconds={LOCKED_SHIFT_ARG}',
        f'--cms-mode {LOCKED_CMS_MODE}',
        f'--cms-ema-decay {LOCKED_CMS_EMA_DECAY}',
        f'--subtraction-etas {LOCKED_SUBTRACTION_ETAS}',
        f'--control-method {LOCKED_CONTROL_METHOD_ARG}',
        f'--control-metric {LOCKED_CONTROL_METRIC_ARG}',
    ]
    if delay is not None:
        cmd_parts.append(f'--common-horizon-delay {int(delay)}')

    run_cmd(' '.join(cmd_parts))

    ok, errs, data = verify_causal_fast_audit_summary(summary_json)
    if not ok or data is None:
        raise RuntimeError(f"任务 {task['task_id']} 产出校验失败: {errs}")

    raw_row = (data.get('negative_control_methods_aggregate') or {}).get('raw') or {}
    cms_row = (data.get('negative_control_methods_aggregate') or {}).get('causal_cms:eta=0.5') or {}
    print(
        f"\n[VERIFIED OK] {task['role']} | N_full={LOCKED_FULL_QUERY_COUNT}, N50={LOCKED_N50_QUERY_COUNT} | "
        f"Raw(Clean={raw_row.get('mean_clean_vocab50_macro_top10', 0)*100:.2f}%, "
        f"Ctrl={raw_row.get('mean_shifted_vocab50_macro_top10', 0)*100:.2f}%, "
        f"Gap={raw_row.get('mean_gap_vocab50_macro_top10', 0)*100:+.2f} pp) | "
        f"CausalCMS_0.5(Clean={cms_row.get('mean_clean_vocab50_macro_top10', 0)*100:.2f}%, "
        f"Ctrl={cms_row.get('mean_shifted_vocab50_macro_top10', 0)*100:.2f}%, "
        f"Gap={cms_row.get('mean_gap_vocab50_macro_top10', 0)*100:+.2f} pp)"
    )
    return data


# ==================== 5. 只读检查点与主方法锚点预检 ====================
print('\n' + '=' * 92)
print(' [Preflight] 只读核验 6 组待补跑检查点 + 1 组已锁定主方法锚点')
print('=' * 92)
for t in CAUSAL_FAST_AUDIT_TASKS:
    ckpts_ok = all(
        (Path(DRIVE_RESULTS_DIR) / DATASET / t['exp_tag'] / t['stage'] / 'all-n10' / f'seed-{s:03d}' / 'best.ckpt').is_file()
        for s in (0, 1, 2)
    )
    sum_p = causal_audit_stage_dir(t['exp_tag'], t['stage']) / f'multi_seed_evaluation_{LOCKED_SPLIT}.json'
    done_ok, _, _ = verify_causal_fast_audit_summary(sum_p)
    print(f" - {t['role']:42s} | ckpts={'OK' if ckpts_ok else 'MISSING'} | audit={'DONE' if done_ok else 'PENDING'}")

anchor_sum_p = causal_audit_stage_dir(LOCKED_ANCHOR_SPEC['exp_tag'], LOCKED_ANCHOR_SPEC['stage']) / f'multi_seed_evaluation_{LOCKED_SPLIT}.json'
anchor_ok, anchor_errs, _ = verify_causal_fast_audit_summary(anchor_sum_p)
print(f" - {LOCKED_ANCHOR_SPEC['role']:42s} | audit={'LOCKED_OK' if anchor_ok else f'CHECK({anchor_errs})'}")
print('\n[OK] 环境、缓存与锁参执行器准备完毕！请按顺序运行后续任务单元格。')


## 单元格 2（任务 1 · 主表基线 1）：`lr4e-4_bs256 / word` (Stage 1 冻结词编码器)


In [ ]:
# 任务 1：主表基线 1 — Stage 1 冻结词编码器 (lr4e-4_bs256 / word)
# 前向编码后按 --common-horizon-delay 2 裁剪至与 Stage 2.3 完全一致的 14,590 / 7,440 查询池
run_causal_fast_audit_task(CAUSAL_FAST_AUDIT_TASKS[0])


## 单元格 3（任务 2 · 主表基线 2a）：`stage1_lr4e-4_bs256_stage2_lr1e-4_bs128 / context` (未去偏全句上下文·微调 CNN)


In [ ]:
# 任务 2：主表基线 2a — 未去偏全句上下文·微调 CNN (stage1_lr4e-4_bs256_stage2_lr1e-4_bs128 / context)
# 直接读取 canonical YAML（eval-run 从 best.ckpt 恢复模型结构与 1.0s 窗口，按 --common-horizon-delay 2 对齐 14,590 / 7,440）
run_causal_fast_audit_task(CAUSAL_FAST_AUDIT_TASKS[1])


## 单元格 4（任务 3 · 主表基线 2b）：`stage1_freeze_bs256_stage2_lr1e-4_bs128 / context` (未去偏全句上下文·冻结 Stage 1)


In [ ]:
# 任务 3：主表基线 2b — 未去偏全句上下文·冻结 Stage 1 (stage1_freeze_bs256_stage2_lr1e-4_bs128 / context)
run_causal_fast_audit_task(CAUSAL_FAST_AUDIT_TASKS[2])


## 单元格 5（任务 4 · 主表基线 3）：`lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window / context` (孪生去偏全句非因果参考)


In [ ]:
# 任务 4：主表基线 3 — 孪生去偏全句非因果参考 (...siamese_debiased_lamda_0.5_history_window / context)
run_causal_fast_audit_task(CAUSAL_FAST_AUDIT_TASKS[3])


## 单元格 6（任务 5 · 核心消融 1）：`lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window / revision` (Stage 2.2 局部固定窗 $R=2$)


In [ ]:
# 任务 5：核心消融 1 — Stage 2.2 局部固定窗 R=2 修订 (...siamese_debiased_lamda_0.5_history_window / revision)
# 原生 revision_availability_masks[2] 即精确定位 N=14,590 / N50=7,440 共同有效查询，无需 --common-horizon-delay
run_causal_fast_audit_task(CAUSAL_FAST_AUDIT_TASKS[4])


## 单元格 7（任务 6 · 核心消融 2）：`lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window / prefix_revision` (Stage 2.3 无 $L_{\Delta\text{-null}}$ 消融)


In [ ]:
# 任务 6：核心消融 2 — Stage 2.3 无 L_delta-null 前缀修订消融 (...siamese_debiased_lamda_0.5_history_window / prefix_revision)
run_causal_fast_audit_task(CAUSAL_FAST_AUDIT_TASKS[5])


## 单元格 8（任务 7 · 可选一键批处理）：按顺序补完所有尚未就绪的 6 组 `causal_fast_audit` 任务


In [ ]:
# 任务 7（可选）：自动依次检查并补跑全部 6 组任务（若已在上方逐格跑完，将自动秒级跳过并校验）
for task_item in CAUSAL_FAST_AUDIT_TASKS:
    run_causal_fast_audit_task(task_item)
print('\n[OK] 全部 6 组 causal_fast_audit 补跑与校验完成！')


## 单元格 9（任务 8 · 收官汇总）：主方法锚点只读联检 + 7 组 `causal_fast_audit` 统一对比总表导出 (`.csv` / `.md` / `.json`)


In [ ]:
# 任务 8：只读加载主方法锚点 + 汇总全部 7 组统一口径结果并导出 CSV / Markdown / JSON 总表
all_specs = CAUSAL_FAST_AUDIT_TASKS + [LOCKED_ANCHOR_SPEC]


def _fmt_pct_mean_sd(mean_v, std_v, *, signed: bool = False) -> str:
    if mean_v is None:
        return '-'
    m_pct = float(mean_v) * 100.0
    s_pct = float(std_v or 0.0) * 100.0
    sign = '+' if signed and m_pct >= 0 else ''
    return f'{sign}{m_pct:.2f} ± {s_pct:.2f}%'


rows = []
all_aligned = True
for spec in all_specs:
    sum_path = causal_audit_stage_dir(spec['exp_tag'], spec['stage']) / f'multi_seed_evaluation_{LOCKED_SPLIT}.json'
    ok, errs, payload = verify_causal_fast_audit_summary(sum_path)
    if not ok or payload is None:
        all_aligned = False
        print(f"[ERROR] {spec['role']} 校验未通过: {errs}")
        continue

    vsa = payload.get('vocabulary_sweep_aggregate') or {}
    n50 = vsa.get('N50') or {}
    full_v = vsa.get('full_vocab') or {}
    ncm = payload.get('negative_control_methods_aggregate') or {}
    raw = ncm.get('raw') or {}
    cms = ncm.get('causal_cms:eta=0.5') or {}

    row = {
        'task_id': spec['task_id'],
        'role': spec['role'],
        'exp_tag': spec['exp_tag'],
        'stage': spec['stage'],
        'seeds': ','.join(str(x) for x in (payload.get('seeds') or [])),
        'n_full_queries': int(full_v.get('query_count') or 0),
        'n50_queries': int(n50.get('query_count') or 0),
        'raw_clean_50w_top10_mean': raw.get('mean_clean_vocab50_macro_top10'),
        'raw_clean_50w_top10_std': raw.get('std_clean_vocab50_macro_top10'),
        'raw_ctrl_50w_top10_mean': raw.get('mean_shifted_vocab50_macro_top10'),
        'raw_ctrl_50w_top10_std': raw.get('std_shifted_vocab50_macro_top10'),
        'raw_gap_50w_top10_mean': raw.get('mean_gap_vocab50_macro_top10'),
        'raw_gap_50w_top10_std': raw.get('std_gap_vocab50_macro_top10'),
        'raw_gap_50w_top10_p': raw.get('p_str_gap_vocab50_macro_top10', '-'),
        'raw_clean_50w_top1_mean': raw.get('mean_clean_vocab50_macro_top1'),
        'raw_clean_full_top10_mean': raw.get('mean_clean_full_vocab_macro_top10'),
        'raw_ctrl_full_top10_mean': raw.get('mean_shifted_full_vocab_macro_top10'),
        'raw_gap_full_top10_mean': raw.get('mean_gap_full_vocab_macro_top10'),
        'cms05_clean_50w_top10_mean': cms.get('mean_clean_vocab50_macro_top10'),
        'cms05_clean_50w_top10_std': cms.get('std_clean_vocab50_macro_top10'),
        'cms05_ctrl_50w_top10_mean': cms.get('mean_shifted_vocab50_macro_top10'),
        'cms05_ctrl_50w_top10_std': cms.get('std_shifted_vocab50_macro_top10'),
        'cms05_gap_50w_top10_mean': cms.get('mean_gap_vocab50_macro_top10'),
        'cms05_gap_50w_top10_std': cms.get('std_gap_vocab50_macro_top10'),
        'cms05_gap_full_top10_mean': cms.get('mean_gap_full_vocab_macro_top10'),
        'summary_json': str(sum_path),
    }
    rows.append(row)

summary_dir = Path(DRIVE_RESULTS_DIR) / DATASET / 'causal_fast_audit_summary'
summary_dir.mkdir(parents=True, exist_ok=True)

csv_path = summary_dir / f'causal_fast_audit_word_main_table_{LOCKED_SPLIT}.csv'
md_path = summary_dir / f'causal_fast_audit_word_main_table_{LOCKED_SPLIT}.md'
json_path = summary_dir / f'causal_fast_audit_word_main_table_{LOCKED_SPLIT}.json'

if rows:
    with csv_path.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

md_lines = [
    f'# Pallier2025 统一 `causal_fast_audit` 词级主表与核心消融对比汇总 ({LOCKED_SPLIT.upper()}, Seeds 0,1,2)',
    '',
    '- 统一锁定口径：`N_full=14,590`、`N50=7,440`、`shift_mode=preroll` (`45.0s`)、`shifts=[-10, -20, -30, -40]`、`cms_mode=causal_ema` (`ema_decay=0.99`, `mu0=train_readonly_fallback`)、`subspace=disabled`',
    '',
    '| 论文角色 | stage | Clean 50w Top-10 | Causal Ctrl (Raw, M=4) | Net Gap (Raw) | Causal CMS (eta=0.5) Clean | Causal CMS (eta=0.5) Ctrl | Causal CMS (eta=0.5) Gap | Full-Vocab Net Gap (Raw) |',
    '| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |',
]
for r in rows:
    md_lines.append(
        '| '
        + ' | '.join(
            [
                r['role'],
                r['stage'],
                _fmt_pct_mean_sd(r['raw_clean_50w_top10_mean'], r['raw_clean_50w_top10_std']),
                _fmt_pct_mean_sd(r['raw_ctrl_50w_top10_mean'], r['raw_ctrl_50w_top10_std']),
                _fmt_pct_mean_sd(r['raw_gap_50w_top10_mean'], r['raw_gap_50w_top10_std'], signed=True),
                _fmt_pct_mean_sd(r['cms05_clean_50w_top10_mean'], r['cms05_clean_50w_top10_std']),
                _fmt_pct_mean_sd(r['cms05_ctrl_50w_top10_mean'], r['cms05_ctrl_50w_top10_std']),
                _fmt_pct_mean_sd(r['cms05_gap_50w_top10_mean'], r['cms05_gap_50w_top10_std'], signed=True),
                _fmt_pct_mean_sd(r['raw_gap_full_top10_mean'], 0.0, signed=True),
            ]
        )
        + ' |'
    )

md_text = '\n'.join(md_lines) + '\n'
md_path.write_text(md_text, encoding='utf-8')
json_path.write_text(
    json.dumps(
        {
            'all_aligned': all_aligned,
            'group_count': len(rows),
            'locked_protocol': {
                'eval_subdir': EVAL_SUBDIR,
                'full_vocab_query_count': LOCKED_FULL_QUERY_COUNT,
                'n50_query_count': LOCKED_N50_QUERY_COUNT,
                'shift_mode': LOCKED_SHIFT_MODE,
                'preroll_seconds': LOCKED_PREROLL_SECONDS,
                'shift_seconds': LOCKED_SHIFT_SECONDS,
                'cms_mode': LOCKED_CMS_MODE,
                'ema_decay': LOCKED_CMS_EMA_DECAY,
                'control_methods': LOCKED_CONTROL_METHODS,
                'subspace_enabled': False,
            },
            'rows': rows,
        },
        ensure_ascii=False,
        indent=2,
    )
    + '\n',
    encoding='utf-8',
)

print(md_text)
print(f'[EXPORTED CSV ] {csv_path}')
print(f'[EXPORTED MD  ] {md_path}')
print(f'[EXPORTED JSON] {json_path}')
if not all_aligned or len(rows) != len(all_specs):
    raise RuntimeError(f'总表汇总未全量通过 (aligned={all_aligned}, rows={len(rows)}/{len(all_specs)})，请检查上方缺失或告警项。')
print('[ALL OK] 全部 7 组模型已严格对齐统一 causal_fast_audit 口径并导出总表！')
